1 . Basic code : request - response

In [8]:
!pip install litellm

# Load API key từ Colab Secret
import os
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key

from litellm import completion
from typing import List, Dict


def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get response"""
    response = completion(
        model="groq/llama-3.1-8b-instant", # Changed model to a currently supported one
        messages=messages,
        max_tokens=1024
    )
    return response.choices[0].message.content


messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)

Here's a function using Python that swaps the keys and values in a dictionary:

```python
from collections import OrderedDict

def swap_dict_items(d):
    """
    Swap the keys and values in a dictionary.

    Args:
        d (dict): The input dictionary.

    Returns:
        dict: A new dictionary with the keys and values swapped.

    Raises:
        ValueError: If the dictionary contains a non-hashable type.
    """
    try:
        # Create a new ordered dictionary to store the swapped items
        swapped = OrderedDict()

        # Iterate over the items in the input dictionary
        for k, v in d.items():
            # Use the original value as the new key and the original key as the new value
            swapped[v] = k

        return swapped
    except TypeError:
        # Raise a ValueError if the dictionary contains a non-hashable type
        raise ValueError("Dictionary contains a non-hashable type")
```

However, due to how dictionaries work in Python, we can't simply 

2. Simple input chatbox

In [9]:

what_to_help_with = input("What do you need help with?")

messages = [
    {"role": "system", "content": "You are a helpful customer service representative. No matter what the user asks, the solution is to tell them to turn their computer or modem off and then back on."},
    {"role": "user", "content": what_to_help_with}
]

response = generate_response(messages)
print(response)

What do you need help with?i love u
That's sweet, but I'm here to help with any technical issues you might be having. If you're experiencing any problems with your internet connection or computer, try turning it off and then back on. This can often resolve connectivity issues.


3 . Smart pipeline for function ai agent : code - doc - test

In [10]:
from litellm import completion
from typing import List, Dict
import sys



def extract_code_block(response: str) -> str:
   """Extract code block from response"""

   if not '```' in response:
      return response

   code_block = response.split('```')[1].strip()
   # Check for "python" at the start and remove

   if code_block.startswith("python"):
      code_block = code_block[6:]

   return code_block

def develop_custom_function():
   # Get user input for function description
   print("\nWhat kind of function would you like to create?")
   print("Example: 'A function that calculates the factorial of a number'")
   print("Your description: ", end='')
   function_description = input().strip()

   # Initialize conversation with system prompt
   messages = [
      {"role": "system", "content": "You are a Python expert helping to develop a function."}
   ]

   # First prompt - Basic function
   messages.append({
      "role": "user",
      "content": f"Write a Python function that {function_description}. Output the function in a ```python code block```."
   })
   initial_function = generate_response(messages)

   # Parse the response to get the function code
   initial_function = extract_code_block(initial_function)

   print("\n=== Initial Function ===")
   print(initial_function)

   # Add assistant's response to conversation
   # Notice that I am purposely causing it to forget its commentary and just see the code so that
   # it appears that is always outputting just code.
   messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})

   # Second prompt - Add documentation
   messages.append({
      "role": "user",
      "content": "Add comprehensive documentation to this function, including description, parameters, "
                 "return value, examples, and edge cases. Output the function in a ```python code block```."
   })
   documented_function = generate_response(messages)
   documented_function = extract_code_block(documented_function)
   print("\n=== Documented Function ===")
   print(documented_function)

   # Add documentation response to conversation
   messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+documented_function+"\n\n\`\`\`"})

   # Third prompt - Add test cases
   messages.append({
      "role": "user",
      "content": "Add unittest test cases for this function, including tests for basic functionality, "
                 "edge cases, error cases, and various input scenarios. Output the code in a \`\`\`python code block\`\`\`."
   })
   test_cases = generate_response(messages)
   # We will likely run into random problems here depending on if it outputs JUST the test cases or the
   # test cases AND the code. This is the type of issue we will learn to work through with agents in the course.
   test_cases = extract_code_block(test_cases)
   print("\n=== Test Cases ===")
   print(test_cases)

   # Generate filename from function description
   filename = function_description.lower()
   filename = ''.join(c for c in filename if c.isalnum() or c.isspace())
   filename = filename.replace(' ', '_')[:30] + '.py'

   # Save final version
   with open(filename, 'w') as f:
      f.write(documented_function + '\n\n' + test_cases)

   return documented_function, test_cases, filename

if __name__ == "__main__":


   function_code, tests, filename = develop_custom_function()
   print(f"\nFinal code has been saved to {filename}")

<>:49: SyntaxWarning: invalid escape sequence '\`'
<>:49: SyntaxWarning: invalid escape sequence '\`'
<>:63: SyntaxWarning: invalid escape sequence '\`'
<>:63: SyntaxWarning: invalid escape sequence '\`'
<>:69: SyntaxWarning: invalid escape sequence '\`'
<>:49: SyntaxWarning: invalid escape sequence '\`'
<>:49: SyntaxWarning: invalid escape sequence '\`'
<>:63: SyntaxWarning: invalid escape sequence '\`'
<>:63: SyntaxWarning: invalid escape sequence '\`'
<>:69: SyntaxWarning: invalid escape sequence '\`'
/tmp/ipykernel_229/33366890.py:49: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})
/tmp/ipykernel_229/33366890.py:49: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})
/tmp/ipykernel_229/33366890.py:63: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "content"


What kind of function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: sum of a and b

=== Initial Function ===

# Define a function to calculate the sum of two numbers
def add_numbers(a, b):
    """
    This function takes two arguments, a and b, and returns their sum.

    Args:
        a (float): The first number.
        b (float): The second number.

    Returns:
        float: The sum of a and b.
    """
    # Check if both a and b are numbers
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        raise ValueError("Both a and b must be numbers.")

    # Calculate and return the sum of a and b
    return a + b

# Example usage:
print(add_numbers(5, 10))  # Output: 15
print(add_numbers(-3, 7))  # Output: 4

=== Documented Function ===

"""
Module for arithmetic operations.

This module provides a function to calculate the sum of two numbers.
"""


def add_numbers(a: int or float, b: int or float